In [1]:
import os
import cv2
import math
import json
import random
import numpy as np
import pandas as pd

from PIL import Image

from random import seed
from torch import nn
from collections import defaultdict
from scipy.ndimage import label
from abc import abstractmethod
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision.transforms as T
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader

from sklearn.metrics import roc_auc_score

from tqdm import tqdm
from IPython.display import Image as IPythonImage

In [ ]:
# Update the root dataset directory based on the university server structure
DATASET_ROOT_DIR = '/home/hafiz/my_thesis/seg_recons/dataset/screw_bag'

# Define the directories for training and test images and masks
TRAIN_IMAGE_DIR = os.path.join(DATASET_ROOT_DIR, 'train/good')
TEST_IMAGE_DIR = os.path.join(DATASET_ROOT_DIR, 'test/logical_anomalies')
VAL_IMAGE_DIR = os.path.join(DATASET_ROOT_DIR, 'validation/good')


# Print the updated paths
print(TRAIN_IMAGE_DIR)
print(TEST_IMAGE_DIR)
print(TEST_IMAGE_DIR)

In [ ]:
# ======================================
# 1. Data Preparation
# ======================================

# Define the custom dataset for MVTec LOCO
class MVTecLOCO(Dataset):
    def __init__(self, root_dir, transform=None, is_train=True):
        """
        Args:
            root_dir (string): Directory with all the images.
            transform (callable, optional): Optional transform to be applied on a sample.
            is_train (bool): True for training data, False for test data.
        """
        self.root_dir = root_dir
        self.transform = transform
        self.is_train = is_train
        self.image_paths = []
        self.labels = []  # 0 for normal, 1 for anomalous

        if self.is_train:
            normal_dir = os.path.join(root_dir, 'train/good')
            for img_name in os.listdir(normal_dir):
                self.image_paths.append(os.path.join(normal_dir, img_name))
                self.labels.append(0)
        else:
            # Test data includes both normal and anomalous images
            test_dir = os.path.join(root_dir, 'test')
            for defect_type in os.listdir(test_dir):
                defect_dir = os.path.join(test_dir, defect_type)
                label = 0 if defect_type == 'good' else 1
                for img_name in os.listdir(defect_dir):
                    self.image_paths.append(os.path.join(defect_dir, img_name))
                    self.labels.append(label)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert('RGB')
        label = self.labels[idx]
        if self.transform:
            image = self.transform(image)
        return image, label

# Define image transformations
transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406],  # Using ImageNet mean and std
                         [0.229, 0.224, 0.225])
])

# Paths to the dataset
root_dir = '/path/to/mvtec_loco_dataset'  # Change this to your dataset path

# Create datasets and dataloaders
train_dataset = MVTecLOCO(root_dir=root_dir, transform=transform, is_train=True)
val_dataset = MVTecLOCO(root_dir=root_dir, transform=transform, is_train=True)
test_dataset = MVTecLOCO(root_dir=root_dir, transform=transform, is_train=False)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)



In [ ]:
# Update the root dataset directory based on the university server structure
DATASET_ROOT_DIR = '/home/hafiz/my_thesis/seg_recons/dataset/black_pak_screws'

# Define the directories for training and test images and masks
TRAIN_IMAGE_DIR = os.path.join(DATASET_ROOT_DIR, 'train/struct_images')
TRAIN_MASK_DIR = os.path.join(DATASET_ROOT_DIR, 'train/struct_masks')
TRAIN_ANNOTATIONS_FILE = os.path.join(DATASET_ROOT_DIR, 'train/train_annotations.csv')
TRAIN_HASANOMALY_FILE = os.path.join(DATASET_ROOT_DIR, 'train/train_hasanomaly.csv')

TEST_IMAGE_DIR = os.path.join(DATASET_ROOT_DIR, 'test/struct_images')
TEST_MASK_DIR = os.path.join(DATASET_ROOT_DIR, 'test/struct_masks')
TEST_ANNOTATIONS_FILE = os.path.join(DATASET_ROOT_DIR, 'test/test_annotations.csv')
TEST_HASANOMALY_FILE = os.path.join(DATASET_ROOT_DIR, 'test/test_hasanomaly.csv')

# Print the updated paths
print(TRAIN_IMAGE_DIR)
print(TRAIN_MASK_DIR)
print(TRAIN_ANNOTATIONS_FILE)
print(TRAIN_HASANOMALY_FILE)

print(TEST_IMAGE_DIR)
print(TEST_MASK_DIR)
print(TEST_ANNOTATIONS_FILE)
print(TEST_HASANOMALY_FILE)

In [3]:
# Dataset class for screw images
class ScrewDataset(Dataset):
    def __init__(self, image_dir, annotations_file, transform=None):
        self.image_dir = image_dir
        self.annotations = pd.read_csv(annotations_file)
        self.transform = transform

    def __len__(self):
        return len(self.annotations)

    def __getitem__(self, idx):
        # Get the image file name and corresponding screw counts
        image_file = self.annotations.iloc[idx, 0]
        small_screw_count = self.annotations.iloc[idx, 1]
        large_screw_count = self.annotations.iloc[idx, 2]

        # Load the image
        img_path = os.path.join(self.image_dir, f"{image_file}.png")
        image = cv2.imread(img_path)

        # Check if the image was loaded successfully
        if image is None:
            raise FileNotFoundError(f"Image {img_path} not found or unable to open.")

        # Convert BGR to RGB
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        
        if self.transform:
            image = self.transform(image)

        return image, small_screw_count, large_screw_count

# Image transformations (Resizing, patching, and normalizing)
def get_preprocessing_transform(image_size=224, patch_size=16):
    transform = transforms.Compose([
        transforms.ToPILImage(),                  # Convert numpy array to PIL image
        transforms.Resize((image_size, image_size)),  # Resize the image
        transforms.ToTensor(),                    # Convert image to tensor
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),  # Normalize
    ])
    return transform

# # Example usage
# if __name__ == "__main__":
#     # Assuming you have your image directory and annotations file
#     image_dir = TRAIN_IMAGE_DIR
#     annotations_file = TRAIN_ANNOTATIONS_FILE

#     # Get the transformation pipeline
#     transform = get_preprocessing_transform()

#     # Create the dataset
#     dataset = ScrewDataset(image_dir, annotations_file, transform=transform)

#     # Create a DataLoader
#     dataloader = DataLoader(dataset, batch_size=2, shuffle=False)

#     # Iterate through the DataLoader
#     for images, small_screw_counts, large_screw_counts in dataloader:
#         print("Batch of images:", images.shape)
#         print("Small screw counts:", small_screw_counts)
#         print("Large screw counts:", large_screw_counts)


In [4]:
from torchvision.models import vit_b_16

class ScrewCountingViT(nn.Module):
    def __init__(self, num_patches=196, hidden_dim=768, num_heads=12, num_classes=2):
        super(ScrewCountingViT, self).__init__()

        # Load the pre-trained Vision Transformer (ViT) base model
        self.vit = vit_b_16(pretrained=True)

        # Modify the ViT to return only the CLS token, which is a summary of the image
        self.vit.heads = nn.Identity()  # Remove the pre-trained classification head

        # Screw count prediction head for small screws
        self.small_screw_head = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 1)  # Output 1 value (count of small screws)
        )

        # Screw count prediction head for large screws
        self.large_screw_head = nn.Sequential(
            nn.Linear(hidden_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 1)  # Output 1 value (count of large screws)
        )

    def forward(self, x):
        # Pass the input image through the ViT model
        x = self.vit(x)  # Extract features from the CLS token

        # Predict small and large screw counts
        small_screw_count = self.small_screw_head(x)
        large_screw_count = self.large_screw_head(x)

        return small_screw_count, large_screw_count

In [ ]:
import logging

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(message)s')

# Define evaluation function
def evaluate_model(model, test_loader, device):
    model.eval()  # Set the model to evaluation mode
    model.to(device)
    
    with torch.no_grad():  # Disable gradient calculations for evaluation
        for i, (images, small_screw_counts, large_screw_counts) in enumerate(test_loader):
            images = images.to(device)
            small_screw_counts = small_screw_counts.float().unsqueeze(1).to(device)
            large_screw_counts = large_screw_counts.float().unsqueeze(1).to(device)

            # Forward pass to get predictions
            pred_small_screws, pred_large_screws = model(images)

            # Get predicted and actual screw counts
            pred_small_screws = pred_small_screws.item()  # Convert to Python number
            pred_large_screws = pred_large_screws.item()  # Convert to Python number
            true_small_screws = small_screw_counts.item()
            true_large_screws = large_screw_counts.item()

            # Determine if the predicted counts are equal
            is_equal_pred = (pred_small_screws == pred_large_screws)
            is_equal_actual = (true_small_screws == true_large_screws)

            # Log the results for each image
            logging.info(f"Image {i + 1}:")
            logging.info(f"  Predicted small screw count: {pred_small_screws:.2f}, Predicted large screw count: {pred_large_screws:.2f}")
            logging.info(f"  Actual small screw count: {true_small_screws:.2f}, Actual large screw count: {true_large_screws:.2f}")
            logging.info(f"  Predicted count equal? {'Yes' if is_equal_pred else 'No'}")
            logging.info(f"  Actual count equal? {'Yes' if is_equal_actual else 'No'}")
            logging.info("-" * 50)

# Define training function
def train_model(model, train_loader, num_epochs=100, learning_rate=0.001, save_model_path='vitmodel.pth'):
    # Move model to GPU if available
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model = model.to(device)

    # Define optimizer and loss function
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)
    criterion = nn.MSELoss()

    # Training loop
    for epoch in range(num_epochs):
        model.train()  # Set model to training mode
        running_loss = 0.0

        for images, small_screw_counts, large_screw_counts in train_loader:
            # Move data to GPU if available
            images = images.to(device)
            small_screw_counts = small_screw_counts.float().unsqueeze(1).to(device)  # Reshape and move to GPU
            large_screw_counts = large_screw_counts.float().unsqueeze(1).to(device)  # Reshape and move to GPU

            # Zero the parameter gradients
            optimizer.zero_grad()

            # Forward pass (get model predictions)
            pred_small_screws, pred_large_screws = model(images)

            # Calculate loss (MSE for both small and large screws)
            loss_small = criterion(pred_small_screws, small_screw_counts)
            loss_large = criterion(pred_large_screws, large_screw_counts)
            loss = loss_small + loss_large

            # Backward pass (compute gradients and update model weights)
            loss.backward()
            optimizer.step()

            # Accumulate loss
            running_loss += loss.item()

        # Print (or log) training loss for the current epoch
        avg_loss = running_loss / len(train_loader)
        logging.info(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

    # Save the trained model
    torch.save(model.state_dict(), save_model_path)
    logging.info("Training completed. Model saved.")

# Example usage
if __name__ == "__main__":
    # Initialize model
    model = ScrewCountingViT()
    transform = get_preprocessing_transform()

    # Define the dataset and DataLoader for training
    train_dataset = ScrewDataset(image_dir=TRAIN_IMAGE_DIR, annotations_file=TRAIN_ANNOTATIONS_FILE, transform=transform)
    train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

    # Train the model
    train_model(model, train_loader, num_epochs=100, learning_rate=0.001)

    # Load the test dataset
    test_dataset = ScrewDataset(image_dir=TEST_IMAGE_DIR, annotations_file=TEST_ANNOTATIONS_FILE, transform=transform)
    test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)  # Set batch_size=1 for printing each image's result

    # Evaluate the model on test data
    evaluate_model(model, test_loader, torch.device("cuda" if torch.cuda.is_available() else "cpu"))

In [ ]:
from sklearn.metrics import mean_squared_error, accuracy_score

def evaluate_model(model, test_loader):
    model.eval()  # Set the model to evaluation mode
    total_mse_small, total_mse_large = 0.0, 0.0
    all_true_anomalies, all_pred_anomalies = [], []

    with torch.no_grad():  # Disable gradient calculations for evaluation
        for images, small_screw_counts, large_screw_counts in test_loader:
            images = images.cuda()  # Move to GPU if available
            small_screw_counts = small_screw_counts.float().unsqueeze(1).cuda()  # Reshape and move to GPU
            large_screw_counts = large_screw_counts.float().unsqueeze(1).cuda()  # Reshape and move to GPU

            # Forward pass (get predictions)
            pred_small_screws, pred_large_screws = model(images)

            # Calculate MSE for screw counts
            mse_small = mean_squared_error(small_screw_counts.cpu().numpy(), pred_small_screws.cpu().numpy())
            mse_large = mean_squared_error(large_screw_counts.cpu().numpy(), pred_large_screws.cpu().numpy())

            total_mse_small += mse_small
            total_mse_large += mse_large

            # Anomaly detection (whether the number of small screws equals large screws)
            true_anomalies = (small_screw_counts.cpu().numpy() != large_screw_counts.cpu().numpy()).astype(int)  # 1 if anomaly, else 0
            pred_anomalies = (pred_small_screws.cpu().numpy() != pred_large_screws.cpu().numpy()).astype(int)

            all_true_anomalies.extend(true_anomalies)
            all_pred_anomalies.extend(pred_anomalies)

    # Calculate average MSE for screw counts
    avg_mse_small = total_mse_small / len(test_loader)
    avg_mse_large = total_mse_large / len(test_loader)

    # Calculate anomaly detection accuracy
    anomaly_accuracy = accuracy_score(all_true_anomalies, all_pred_anomalies)

    print(f"Average MSE for Small Screws: {avg_mse_small:.4f}")
    print(f"Average MSE for Large Screws: {avg_mse_large:.4f}")
    print(f"Anomaly Detection Accuracy: {anomaly_accuracy:.4f}")


In [ ]:
# ViT for counting anomaly detection
class ViTCountingAnomaly(nn.Module):
    def __init__(self, image_size=256, patch_size=16, dim=512, depth=6, heads=8, mlp_dim=1024):
        super(ViTCountingAnomaly, self).__init__()
        assert image_size % patch_size == 0, "Image size must be divisible by patch size."

        self.num_patches = (image_size // patch_size) ** 2
        self.patch_dim = patch_size * patch_size * 3

        self.patch_embedding = nn.Linear(self.patch_dim, dim)

        self.pos_embedding = nn.Parameter(torch.randn(1, self.num_patches + 1, dim))
        self.cls_token = nn.Parameter(torch.randn(1, 1, dim))
        
        self.transformer = nn.ModuleList([
            nn.TransformerEncoderLayer(
                d_model=dim, nhead=heads, dim_feedforward=mlp_dim
            ) for _ in range(depth)
        ])
        
        self.to_cls_token = nn.Identity()

        # Two separate outputs: one for small screws and one for large screws
        self.mlp_head_small = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.ReLU(),
            nn.Linear(mlp_dim, 1)  # Predicts count for small screws
        )

        self.mlp_head_large = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, mlp_dim),
            nn.ReLU(),
            nn.Linear(mlp_dim, 1)  # Predicts count for large screws
        )

    def forward(self, x):
        B, C, H, W = x.shape
        patches = x.unfold(2, 16, 16).unfold(3, 16, 16)
        patches = patches.contiguous().view(B, -1, 16 * 16 * C)
        
        tokens = self.patch_embedding(patches)
        cls_tokens = self.cls_token.expand(B, -1, -1)
        x = torch.cat((cls_tokens, tokens), dim=1)
        x += self.pos_embedding

        for transformer_layer in self.transformer:
            x = transformer_layer(x)

        x = self.to_cls_token(x[:, 0])
        
        # Predict counts for small and large screws separately
        small_screw_count = self.mlp_head_small(x)
        large_screw_count = self.mlp_head_large(x)

        return small_screw_count, large_screw_count
    
# Verify the ViTCountingAnomaly class
if __name__ == "__main__":
    # Set up parameters
    image_size = 512
    patch_size = 16
    dim = 512
    depth = 6
    heads = 8
    mlp_dim = 1024

    # Create a ViTCountingAnomaly model
    model = ViTCountingAnomaly(image_size=image_size, patch_size=patch_size, dim=dim, depth=depth, heads=heads, mlp_dim=mlp_dim)

    # Create a dummy input (batch_size, channels, height, width)
    dummy_input = torch.randn(1, 3, image_size, image_size)  # Batch size 1, 3-channel image of 256x256

    # Run the model
    small_screw_count, large_screw_count = model(dummy_input)

    # Verify the outputs
    print("Predicted small screw count:", small_screw_count.item())
    print("Predicted large screw count:", large_screw_count.item())

    # Check output shapes
    print("Shape of small screw count:", small_screw_count.shape)  # Should be [1, 1]
    print("Shape of large screw count:", large_screw_count.shape)  # Should be [1, 1]

In [ ]:
def eval(testing_dataset_loader, args, vit_model, data_len, device):
    vit_model.eval()

    tbar = tqdm(testing_dataset_loader)
    
    for i, sample in enumerate(tbar):
        image = sample["image"].to(device)
        small_screw_count = sample['small_screw_count'].to(device)
        large_screw_count = sample['large_screw_count'].to(device)
        
        # Predict screw counts using ViT
        pred_small_screw_count, pred_large_screw_count = vit_model(image)

In [ ]:
def save_models(unet_model, seg_model, vit_model, args, final, epoch, sub_class):
    """
    Save the models (U-Net, Segmentation, and ViT).
    """
    os.makedirs(f'{args["output_path"]}/model/diff-params-ARGS={args["arg_num"]}/{sub_class}', exist_ok=True)
    torch.save(
        {
            'n_epoch': epoch,
            'unet_model_state_dict': unet_model.state_dict(),
            'seg_model_state_dict': seg_model.state_dict(),
            'vit_model_state_dict': vit_model.state_dict(),
            "args": args
        }, f'{args["output_path"]}/model/diff-params-ARGS={args["arg_num"]}/{sub_class}/params-{final}.pt'
    )

In [ ]:
def weights_init(m):
  classname = m.__class__.__name__
  if classname.find('Conv') != -1:
    m.weight.data.normal_(0.0, 0.02)
  elif classname.find('BatchNorm') != -1:
    m.weight.data.normal_(1.0, 0.02)
    m.bias.data.fill_(0)

#del images, labels, noise_loss, pred_x0  # Or other variables you no longer need
torch.cuda.empty_cache()


In [ ]:
def train(training_dataset_loader, testing_dataset_loader, testing_dataset_len, args, device):
    in_channels = args["channels"]

    # Vision Transformer for counting anomalies (ViTCountingAnomaly)
    vit_model = ViTCountingAnomaly(image_size=args['img_size'][0], patch_size=16).to(device)

    # Optimizers
    optimizer_vit = optim.Adam(vit_model.parameters(), lr=args['vit_lr'], weight_decay=args['weight_decay'])

    # Loss functions
    mse_loss = nn.MSELoss()

    # Scheduler
    #scheduler_seg = optim.lr_scheduler.CosineAnnealingLR(optimizer_seg, T_max=10, eta_min=0, verbose=False)

    tqdm_epoch = range(0, args['EPOCHS'])
    
    best_epoch = 0

    for epoch in tqdm_epoch:
        vit_model.train()
        
        train_loss = 0.0
        train_count_loss = 0.0

        tbar = tqdm(training_dataset_loader)
        
        for i, sample in enumerate(tbar):
            aug_image = sample['image'].to(device)
            anomaly_mask = sample['mask'].to(device)
            anomaly_label = sample['has_anomaly'].to(device).squeeze()
            small_screw_count = sample['small_screw_count'].to(device)
            large_screw_count = sample['large_screw_count'].to(device)


            # Count anomalies using ViT (small and large screw counts)
            pred_small_screw_count, pred_large_screw_count = vit_model(aug_image)

            # Calculate losses
            small_screw_loss = mse_loss(pred_small_screw_count, small_screw_count)
            large_screw_loss = mse_loss(pred_large_screw_count, large_screw_count)
            total_loss = small_screw_loss + large_screw_loss

            optimizer_vit.zero_grad()

            total_loss.backward()

            optimizer_vit.step()


            train_loss += total_loss.item()
            train_count_loss += (small_screw_loss + large_screw_loss).item()

            tbar.set_description(f'Epoch: {epoch}, Train Loss: {train_loss:.3f}')
            tbar.set_description(f'Epoch: {epoch}, Train Loss: {train_count_loss:.3f}')

        # Evaluate and save best models periodically
        if (epoch + 1) % 2 == 0 and epoch > 0:
            temp_image_auroc, temp_pixel_auroc = eval(testing_dataset_loader, args, unet_model, ddpm_sample, seg_model, vit_model, data_len, device)
            iou_score = calculate_iou(pred_mask, anomaly_mask)

            tqdm.write(f'Epoch {epoch}: Temp_Image_AUROC: {temp_image_auroc}')
            tqdm.write(f'Epoch {epoch}: Temp_Pixel_AUROC: {temp_pixel_auroc}')
            tqdm.write(f'Epoch {epoch}: IOU_SCORE: {iou_score}')

            if (temp_image_auroc + temp_pixel_auroc) >= (best_image_auroc + best_pixel_auroc):
                if temp_image_auroc >= best_image_auroc:
                    save_models(unet_model, seg_model, vit_model, args, 'best', epoch)
                    best_image_auroc = temp_image_auroc
                    best_pixel_auroc = temp_pixel_auroc
                    best_iou = iou_score
                    best_epoch = epoch

    save_models(unet_model, seg_model, vit_model, args, 'last', epoch)
    print(best_epoch)
    print(best_iou)

def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    args = {
        "img_size": [512,512],
        "Batch_Size": 4,
        "EPOCHS": 3000,
        "T": 1000,
        "base_channels": 128,
        "beta_schedule": "linear",
        "loss_type": "l2",
        "diffusion_lr": 1e-4,
        "seg_lr": 1e-5,
        "random_slice":True,
        "weight_decay": 0.0,
        "save_imgs": True,
        "save_vids":False, 
        "dropout":0,
        "attention_resolutions":"32,16,8",
        "num_heads":4,
        "num_head_channels":-1,
        "noise_fn":"gauss",
        "channels":3,
        "mvtec_root_path":"datasets/mvtec",
        "visa_root_path":"datasets/VisA_1class/1cls",
        "dagm_root_path":"datasets/dagm",
        "mpdd_root_path":"datasets/mpdd",
        "anomaly_source_path":"datasets/DTD",
        "noisier_t_range":600,
        "less_t_range":300,
        "condition_w":1,
        "eval_normal_t":200,
        "eval_noisier_t":400,
        "output_path":"outputs"
    }

    # Custom mask transformation to ensure values remain unchanged
    def mask_to_tensor(mask_image):
        mask_array = np.array(mask_image)
        mask_tensor = torch.tensor(mask_array, dtype=torch.long)
        mask_tensor = mask_tensor.unsqueeze(0)
        return mask_tensor

    # Define the transformations
    image_transform = transforms.Compose([
        transforms.ToTensor(),
        ])

    mask_transform = transforms.Compose([
        transforms.Lambda(mask_to_tensor)
    ])

    training_dataset = ScrewDataset(image_dir=TRAIN_IMAGE_DIR, mask_dir=TRAIN_MASK_DIR, annotations_file = TRAIN_ANNOTATIONS_FILE, image_transform=image_transform, mask_transform=mask_transform)
    training_dataset_loader = DataLoader(training_dataset, batch_size=2,drop_last=True)

    testing_dataset = ScrewDataset(image_dir=TEST_IMAGE_DIR, mask_dir=TEST_MASK_DIR, annotations_file= TEST_ANNOTATIONS_FILE, image_transform=image_transform, mask_transform=mask_transform)
    testing_dataset_loader = DataLoader(testing_dataset, batch_size=1, shuffle=False)

    testing_dataset_len = len(testing_dataset)

    train(training_dataset_loader, testing_dataset_loader, testing_dataset_len, args, device)

if __name__ == '__main__':
    seed(42)
    main()